# Uncertainty-Aware Pneumothorax Segmentation & Selective Prediction
### Primary Benchmark: ResNet34 U-Net on SIIM-ACR
**Target**: MIDL / MICCAI Research Portfolio

This notebook executes the training and evaluation pipelines for:
- **EXP-03**: Deterministic ResNet34 U-Net Baseline
- **EXP-04**: Monte Carlo Dropout (=20$)
- **EXP-05**: Deep Ensemble (=5$)
- **EXP-06, 07, 08**: Calibration, Error Detection AUROC, and Selective Prediction (Risk-Coverage Pareto Curves)


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Mount repository or set working directory
# In Kaggle: ensure 'src' is in PYTHONPATH
sys.path.append('.')
from src.utils.seed import set_seed
from src.utils.rle import rle_decode, aggregate_rle_masks
from src.losses.combined import CombinedBCEDiceLoss
from src.models.unet import PneumothoraxUNet
from src.models.ensemble import DeepEnsemble
from src.metrics.segmentation import compute_dice_coefficient, compute_hausdorff95
from src.metrics.uncertainty import compute_esce, compute_brier_score, compute_error_detection_auroc
from src.metrics.selective_prediction import aggregate_case_uncertainty, compute_risk_coverage_curve, compute_aurc

set_seed(42)
print("Source modules imported successfully.")


In [ ]:
# Load zero-leakage patient-grouped split manifest
SPLITS_CSV = 'data/processed/patient_splits.csv'
df_splits = pd.read_csv(SPLITS_CSV)
print(f"Loaded manifest with {len(df_splits)} images across {df_splits['PatientID'].nunique()} patients.")
print("Fold counts:", df_splits['Fold'].value_counts().to_dict())


In [ ]:
from src.data.dataset import PneumothoraxDataset, get_training_transforms, get_validation_transforms

# Configure training parameters
IMAGE_SIZE = 512
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 35
LR = 3e-4

print(f"Training configured for {IMAGE_SIZE}x{IMAGE_SIZE} at batch size {BATCH_SIZE}.")


## 1. Train EXP-03: Deterministic Baseline
Standard ResNet34 U-Net with bash.5 \cdot 	ext{BCE} + 0.5 \cdot 	ext{SoftDice}$ loss.


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scaler, epochs=35, device='cuda'):
    best_dice = 0.0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            optimizer.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    logits = model(images)
                    loss = criterion(logits, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = criterion(logits, masks)
                loss.backward()
                optimizer.step()
            train_loss += loss.item() * len(images)
        train_loss /= len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f}")
    return model


## 2. Selective Prediction & Clinical Triage Evaluation
Evaluates retained cohort Dice as a function of coverage  \in [0.2, 1.0]$ using Top-500 pixel variance.


In [ ]:
# Benchmark function for computing Risk-Coverage curves
def run_selective_prediction_benchmark(case_uncertainties, case_dices, method_name='Method'):
    case_risks = 1.0 - case_dices
    coverages, risks = compute_risk_coverage_curve(case_uncertainties, case_risks)
    aurc = compute_aurc(coverages, risks)
    print(f"=== {method_name} Selective Prediction ===")
    print(f"AURC (Area Under Risk-Coverage): {aurc:.4f}")
    for cov in [0.70, 0.80, 0.90, 1.00]:
        cutoff = max(1, int(round(cov * len(case_uncertainties))))
        idx = np.argsort(case_uncertainties)[:cutoff]
        ret_dice = float(np.mean(case_dices[idx]))
        print(f"  Retained Dice at {int(cov*100)}% coverage: {ret_dice:.4f}")
    return coverages, risks, aurc
